In [ ]:
#| default_exp core

# fastmux's source

> Drive and inspect tmux from Python: live session, window, and pane handles with CLI-style reprs

## Setup

In [ ]:
#| export
from fastcore.utils import *
from fastcore.meta import delegates

import re, shlex, socket, subprocess, sys, time, uuid

In [ ]:
#| hide
from fastcore.test import *
from nbdev import show_doc

## Running tmux

Everything fastmux does happens by shelling out to the `tmux` binary, which is the one true API to a tmux server. The foundation is a small runner that executes one tmux command and either returns its stdout or raises.

In [ ]:
#| export
class TmuxError(RuntimeError):
    "Raised when a tmux command fails or a target cannot be resolved."

def _tmux(*args: str, input: str | None = None):
    proc = subprocess.run(["tmux", *args], input=input, text=True, capture_output=True, check=False)
    if proc.returncode != 0:
        stderr = proc.stderr.strip() or proc.stdout.strip()
        raise TmuxError(stderr or f"tmux command failed: {args!r}")
    return proc.stdout.rstrip("\n")

`_tmux` takes the command as separate arguments (no shell involved), passes optional `input` on stdin (used later for loading paste buffers), and strips the trailing newline tmux prints. Any failure becomes a `TmuxError` carrying tmux's own message. `tmux -V` needs no running server, so it makes a good first check:

In [ ]:
ver = _tmux('-V')
ver

'tmux 3.7b'

In [ ]:
test_eq(ver.startswith('tmux'), True)
with expect_fail(TmuxError, contains='fastmux-empty'): _tmux('-L','fastmux-empty','ls')

## Querying tmux

tmux answers questions through [format strings](https://man.openbsd.org/tmux#FORMATS): every `list-*` and `display-message` command takes `-F` with `#{variable}` placeholders. So fastmux describes each concept once, as a mapping of field name to format variable, and parses the tab-separated answer into a typed `dict`.

Sessions, windows, and panes all have server-unique ids (`$1`, `@1`, `%1`) which never change or get reused. Names and indexes are ambiguous as `-t` targets (a session auto-named `5` can resolve as a window index!), so fastmux keys every object on its id and only ever targets ids internally.

In [ ]:
#| export
_sess_f = dict(id='session_id', name='session_name', n_wins='session_windows', attached='session_attached')
_win_f  = dict(id='window_id', session='session_name', idx='window_index', name='window_name', active='window_active', n_panes='window_panes')
_pane_f = dict(id='pane_id', session='session_name', window_id='window_id', win='window_index', idx='pane_index',
    cmd='pane_current_command', active='pane_active', dead='pane_dead', width='pane_width', height='pane_height',
    hist='history_size', cy='cursor_y', dead_status='pane_dead_status', title='pane_title')
_ints  = {'win','idx','width','height','hist','cy','n_wins','n_panes'}
_bools = {'active','dead','attached'}

def _fmt(fields): return '\t'.join(f'#{{{v}}}' for v in fields.values())

def _parse(fields, line):
    d = dict(zip(fields, line.split('\t', len(fields)-1)))
    for k in fields.keys() & _ints: d[k] = int(d[k])
    for k in fields.keys() & _bools: d[k] = d[k]=='1'
    return d

def _get(fields, target):
    out = _tmux('display-message','-p','-t',str(target),_fmt(fields))
    if not out.strip('\t'): raise TmuxError(f"can't find {target}")
    return _parse(fields, out)
def _list(cmd, fields, *flags): return [_parse(fields, l) for l in _tmux(cmd,*flags,'-F',_fmt(fields)).splitlines()]

A field spec is used twice: `_fmt` renders it as the `-F` argument, and `_parse` types the reply. The pane title comes last since it is the one field that may itself contain tabs. Some tmux versions report an unknown `-t` target as success with every field empty rather than as an error, so `_get` turns that reply into the `TmuxError` it should have been.

In [ ]:
test_eq(_fmt(dict(id='pane_id', w='pane_width')), '#{pane_id}\t#{pane_width}')
test_eq(_parse(dict(id='pane_id', width='pane_width', active='pane_active'), '%3\t80\t1'),
        dict(id='%3', width=80, active=True))

## Sessions

A *session* is tmux's top-level unit: a named group of windows with its own lifecycle, which clients attach to. All fastmux objects follow the same pattern: an `AttrDict` holding the fields queried from tmux, a `fetch` classmethod that builds one from any target, and `refresh` to re-query in place. The repr matches what `tmux ls` prints.

In [ ]:
#| export
class Session(AttrDict):
    "A tmux session; fields from `_sess_f`, keyed on the immutable session id"
    @classmethod
    def fetch(cls, target): return cls(_get(_sess_f, target))
    def refresh(self):
        self.update(_get(_sess_f, self.id))
        return self
    def __repr__(self): return f"{self.name}: {self.n_wins} windows{' (attached)' if self.attached else ''}"
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

def new_session(
    cmd=None,     # Command to run (str or argv list); default: the user's shell
    name=None,    # Session name; default: tmux auto-numbers
    width=None,   # Terminal width in columns
    height=None,  # Terminal height in rows
    cwd=None,     # Working directory
    env=None,     # Extra environment vars as a dict
    remain=False, # Keep dead panes around (inspectable, with exit status) instead of destroying them
):
    "Start a detached tmux session and return its `Session`"
    args = ['new-session','-d','-P','-F','#{session_id}']
    if name: args += ['-s',name]
    if width: args += ['-x',str(width)]
    if height: args += ['-y',str(height)]
    if cwd: args += ['-c',str(cwd)]
    for k,v in (env or {}).items(): args += ['-e',f'{k}={v}']
    if cmd: args += [cmd] if isinstance(cmd,str) else list(cmd)
    sid = _tmux(*args)
    if remain: _tmux('set-option','-w','-t',sid,'remain-on-exit','on')
    return Session.fetch(sid)

Sessions are created detached (`-d`), so making one never steals the terminal you are working in. We'll create one now running a small script that prints numbered lines then waits. It will be the playground for the rest of this notebook.

In [ ]:
printer = [sys.executable,'-u','-c','import time\n'
                                    'for i in range(30): print(f"ln {i}")\n'
                                    'time.sleep(600)']
s = new_session(printer, width=80, height=10)
s

<div class="prose" markdown="1">

```python
0: 1 windows
```

</div>

In [ ]:
assert s.id.startswith('$') and s.n_wins==1
test_eq(s.refresh().id, s.id)

In [ ]:
#| export
@patch
def kill(self:Session):
    "Kill this session and everything in it"
    sys.audit('fastmux.Session.kill', self.id)
    _tmux('kill-session','-t',self.id)

@patch
def rename(self:Session, name):
    "Rename this session, returning it"
    _tmux('rename-session','-t',self.id,name)
    return self.refresh()

@patch(as_prop=True)
def attach_command(self:Session):
    "Shell command that attaches a terminal to this session"
    return f'tmux attach -t {shlex.quote(self.name)}'

Destructive operations raise a `sys.audit` event, so audit-hook based sandboxes can gate them. In-place operations return `self` for chaining.

In [ ]:
test_eq(s.rename('fastmux-demo').name, 'fastmux-demo')
test_eq(s.attach_command, 'tmux attach -t fastmux-demo')

## Panes

A *pane* is one terminal: a pty running a command, with a visible screen and a scrollback transcript. Everything you can read or type happens in a pane. Because a pane knows its geometry and cursor, we can treat its whole transcript (history plus visible screen) as a list of lines, addressed absolutely from line 0.

In [ ]:
#| export
class Pane(AttrDict):
    "A tmux pane; fields from `_pane_f`, keyed on the immutable pane id"
    @classmethod
    def fetch(cls, target): return cls(_get(_pane_f, target))
    def refresh(self):
        self.update(_get(_pane_f, self.id))
        return self
    @property
    def target(self): return f'{self.session}:{self.win}.{self.idx}'
    @property
    def running(self): return not self.dead
    @property
    def exit_code(self): return int(self.dead_status) if self.dead and self.dead_status else None

In [ ]:
p = Pane.fetch(s.id)
test_eq((p.width, p.height), (80, 10))
assert p.id.startswith('%') and p.running and p.exit_code is None
with expect_fail(TmuxError, contains="can't find"): Pane.fetch('%9999')
p.target

'fastmux-demo:1.1'

`target` is the pane's address in standard tmux syntax (`session:window.pane`), so it can be pasted into any `tmux -t` command.

In [ ]:
#| export
def _nlines(p):
    "Transcript line count up to the cursor, ignoring a trailing blank cursor row"
    n = p.hist + p.cy + 1
    if n <= 0: return 0
    return n-1 if _tmux('capture-pane','-p','-t',p.id,'-S',str(p.cy),'-E',str(p.cy))=='' else n

class Capture(AttrDict):
    "Captured transcript lines plus their absolute range and source pane"
    def __repr__(self):
        status = '' if self.running else (f' · exited {self.exit_code}' if self.exit_code is not None else ' · dead')
        src = f'── {self.target} {self.id} · lines {self.start}-{self.end} of {self.n}{status}'
        return f'{self.text}\n{src}' if self.text else src
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

@patch
def capture(self:Pane, start=0, end=None, ansi=False):
    "A `Capture` of transcript lines `start:end` (absolute; `end=None` means up to the cursor)"
    self.refresh()
    n = _nlines(self)
    if end is None: end = n
    start, end = max(0,min(start,n)), max(0,min(end,n))
    if start >= end: text = ''
    else: text = _tmux( 'capture-pane','-p',*(['-e'] if ansi else []),'-t',self.id,
                        '-S',str(start-self.hist),'-E',str(end-self.hist-1))
    return Capture(text=text, lines=tuple(text.splitlines()), start=start, end=end, n=n,
                   target=self.target, id=self.id, running=self.running, exit_code=self.exit_code)

tmux numbers capture lines relative to the top of the visible screen (scrollback is negative), so `capture` converts from absolute transcript lines. The `Capture` repr is the text itself with a source footer:

In [ ]:
time.sleep(0.5)
c = p.capture(5, 8)
c

<div class="prose" markdown="1">

```python
ln 5
ln 6
ln 7
── fastmux-demo:1.1 %0 · lines 5-8 of 30
```

</div>

In [ ]:
test_eq(c.lines, ('ln 5','ln 6','ln 7'))
test_eq((c.start, c.end, c.n), (5, 8, 30))
test_eq(p.capture(29, 99).lines, ('ln 29',))  # ranges are clamped

In [ ]:
#| export
@patch(as_prop=True)
def text(self:Pane):
    "The visible screen as plain text"
    return _tmux('capture-pane','-p','-t',self.id)

@patch(as_prop=True)
def ansi(self:Pane):
    "The visible screen with ANSI escape sequences"
    return _tmux('capture-pane','-p','-e','-t',self.id)

@patch
def __repr__(self:Pane):
    try: return self.text
    except TmuxError: return f"Pane({self['id']} gone)"

@patch
def _repr_pretty_(self:Pane, pr, cycle): pr.text(repr(self))

A pane's repr is its current screen, so displaying one is like glancing at that terminal:

In [ ]:
p

<div class="prose" markdown="1">

```python
ln 21
ln 22
ln 23
ln 24
ln 25
ln 26
ln 27
ln 28
ln 29
```

</div>

In [ ]:
#| export
@patch
def __len__(self:Pane): return _nlines(self.refresh())

@patch
def __getitem__(self:Pane, i):
    "Transcript line `i` as a str, a `Capture` for a slice; str keys keep normal dict access"
    if isinstance(i, str): return dict.__getitem__(self, i)
    n = len(self)
    if isinstance(i, slice):
        start, end, step = i.indices(n)
        if step != 1: raise ValueError('step slicing not supported')
        return self.capture(start, end)
    if i < 0: i += n
    if not 0 <= i < n: raise IndexError(i)
    return self.capture(i, i+1).text

_seen = {}

@patch
def display(self:Pane, lines=80, ansi=False):
    "Capture the last `lines` transcript lines, recording the pane's last-seen state"
    n = _nlines(self.refresh())
    c = self.capture(max(0, n-lines), ansi=ansi)
    if not ansi: _seen[self.id, lines] = (c.text, c.n, c.running, c.exit_code)
    return c

Indexing and slicing read the transcript like a list of lines, in the same coordinates tmux itself uses, so `p[-200:]` is the last 200 lines including scrollback. Since `Pane` is a dict underneath, string keys still do dict lookup (`p['id']`), while ints and slices read the transcript.

In [ ]:
test_eq(len(p), 30)
test_eq((p[0], p[-1]), ('ln 0', 'ln 29'))
test_eq(p[5:7].lines, ('ln 5','ln 6'))
test_eq(p[-3:].lines, ('ln 27','ln 28','ln 29'))
test_eq(p['id'], p.id)
with expect_fail(IndexError): p[99]

### Sending input

fastmux writes to a pane two ways: `send` pastes literal text through a tmux buffer (no key-name interpretation, so any characters are safe), and `send_keys` sends [tmux key names](https://man.openbsd.org/tmux#KEY_BINDINGS) like `Enter` or `C-c`. Both then `poll`, waiting up to `wait_ms` for the pane to differ from its last-seen state and returning the latest `Capture`, so the common send-then-read round trip is one call.

In [ ]:
#| export
def _snapshot(p, lines):
    c = p.display(lines)
    return c.text, c.n, c.running, c.exit_code

@patch
def poll(self:Pane, wait_ms=0, interval_ms=50, lines=80):
    "Wait up to `wait_ms` for the pane to differ from its last-seen state (returning at once if it already does), then capture the last `lines`"
    if wait_ms > 0:
        base = _seen.get((self.id, lines)) or _snapshot(self, lines)
        deadline = time.monotonic() + wait_ms/1000
        while _snapshot(self, lines) == base and time.monotonic() < deadline:
            time.sleep(min(interval_ms/1000, max(0, deadline-time.monotonic())))
    return self.display(lines)

Let's drive a stdin-echo process end to end, sending text and getting the acknowledgement back in the same call:

In [ ]:
echoer = new_session([sys.executable,'-u','-c',
                      "import sys\n"
                      "print('ready')\n"
                      "for l in sys.stdin: print(f'ACK:{l.rstrip()}')"],
                      width=60, height=8, remain=True)
ep = Pane.fetch(echoer.id)
ep.poll(wait_ms=1500, lines=3)

<div class="prose" markdown="1">

```python
ready
── 1:1.1 %1 · lines 0-1 of 1
```

</div>

In [ ]:
#| export
@patch
def send(self:Pane, chars='', wait_ms=0, interval_ms=50, lines=80):
    "Paste `chars` into this pane literally, then `poll`"
    if chars:
        buf = f'fastmux-{uuid.uuid4().hex}'
        _tmux('load-buffer','-b',buf,'-',input=chars)
        try: _tmux('paste-buffer','-d','-b',buf,'-t',self.id)
        except Exception:
            try: _tmux('delete-buffer','-b',buf)
            except TmuxError: pass
            raise
    return self.poll(wait_ms, interval_ms, lines)

@patch
def send_keys(self:Pane, *keys, wait_ms=0, interval_ms=50, lines=80):
    "Send tmux key names to this pane, then `poll`"
    if keys: _tmux('send-keys','-t',self.id,*keys)
    return self.poll(wait_ms, interval_ms, lines)

@patch
def interrupt(self:Pane, wait_ms=0, interval_ms=50, lines=80):
    "Send `Ctrl-C` to this pane, then `poll`"
    return self.send_keys('C-c', wait_ms=wait_ms, interval_ms=interval_ms, lines=lines)

@patch
def wait(self:Pane, timeout_ms=None, interval_ms=50):
    "Wait for this pane's command to exit, returning its status (`None` on timeout)"
    deadline = None if timeout_ms is None else time.monotonic() + max(timeout_ms,0)/1000
    while True:
        self.refresh()
        if self.dead: return self.exit_code
        if deadline is not None and time.monotonic() >= deadline: return None
        time.sleep(interval_ms/1000)

In [ ]:
r = ep.send('hello\n', wait_ms=1500, lines=3)
assert 'ACK:hello' in r.text
test_eq(ep.wait(timeout_ms=1), None)  # still running

When the process exits, the pane dies but (with `remain-on-exit` set) stays inspectable, and `wait` returns the exit status:

In [ ]:
ep.send_keys('C-d')  # EOF ends the loop
test_eq(ep.wait(timeout_ms=3000), 0)
echoer.kill()

`poll` doesn't just wait for the pane to change after the call: it waits for the pane to differ from the *last state a capture showed you*. Each `display` (and so each `send`, `send_keys`, `interrupt`, and `poll`, which all return one) records the pane's last-seen state, and `poll` returns as soon as the pane differs from it. So output that arrived while you weren't watching satisfies the next `poll` immediately, rather than making it wait for yet another change:

In [ ]:
late = new_session([sys.executable,'-u','-c','import time\n'
                                          'print("early")\n'
                                          'time.sleep(2)\n'
                                          'print("late")\n'
                                          'time.sleep(600)'], width=60, height=8)
lp = Pane.fetch(late.id)
c = lp.poll(wait_ms=500)
assert 'late' not in c.text
c

In [ ]:
time.sleep(2.5)  # "late" arrives while we aren't watching
t0 = time.monotonic(); c = lp.poll(wait_ms=5000); elapsed = time.monotonic()-t0
late.kill()
assert 'late' in c.text
assert elapsed < 1

### Listing panes

In [ ]:
#| export
def _pane_row(p):
    tag = ('' if not p.active else ' (active)') + ('' if not p.dead else ' (dead)')
    title = f' · {p.title}' if p.title and p.title != p.cmd and p.title != socket.gethostname() else ''
    return f'{p.win}.{p.idx}: [{p.width}x{p.height}] {p.id} {p.cmd}{tag}{title}'

class Panes(L):
    "Panes shown as `tmux list-panes`-style summary lines"
    def __repr__(self): return '\n'.join(_pane_row(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

@patch(as_prop=True)
def panes(self:Session):
    "All panes in this session"
    return Panes(Pane(d) for d in _list('list-panes', _pane_f, '-s','-t',self.id))

@patch(as_prop=True)
def pane(self:Session):
    "This session's active pane"
    return Pane.fetch(self.id)

A `Panes` collection reprs as summary lines rather than screens (a dozen full screens would be unreadable). The line format mirrors `tmux list-panes`, with the pane title appended when it carries information (tmux defaults it to the hostname, which doesn't):

In [ ]:
s.panes

1.1: [80x10] %0 python (active)

In [ ]:
test_eq(len(s.panes), 1)
test_eq(s.pane.id, p.id)

## Windows

A *window* groups panes into one screenful, like tabs in a terminal emulator. Its repr is its `tmux list-windows`-style summary line followed by its panes.

In [ ]:
#| export
class Window(AttrDict):
    "A tmux window; fields from `_win_f`, keyed on the immutable window id"
    @classmethod
    def fetch(cls, target): return cls(_get(_win_f, target))
    def refresh(self):
        self.update(_get(_win_f, self.id))
        return self
    @property
    def panes(self): return Panes(Pane(d) for d in _list('list-panes', _pane_f, '-t',self.id))
    def __repr__(self):
        head = f"{self.idx}: {self.name}{'*' if self.active else ''} ({self.n_panes} panes) {self.id}"
        return '\n'.join([head] + [f'  {_pane_row(o)}' for o in self.panes])
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

class Windows(L):
    "Windows shown with their pane listings"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

@patch(as_prop=True)
def windows(self:Session):
    "This session's windows"
    return Windows(Window(d) for d in _list('list-windows', _win_f, '-t',self.id))

@patch
def new_window(
    self:Session,
    cmd=None,     # Command to run (str or argv list); default: the user's shell
    name=None,    # Window name; default: tmux auto-names from the command
    cwd=None,     # Working directory
    env=None,     # Extra environment vars as a dict
    remain=False, # Keep dead panes around instead of destroying them
    focus=False,  # Make it the session's current window
):
    "Create a window in this session, returning the new `Window`"
    args = ['new-window','-P','-F','#{window_id}','-t',self.id]
    if not focus: args += ['-d']
    if name: args += ['-n',name]
    if cwd: args += ['-c',str(cwd)]
    for k,v in (env or {}).items(): args += ['-e',f'{k}={v}']
    if cmd: args += [cmd] if isinstance(cmd,str) else list(cmd)
    wid = _tmux(*args)
    if remain: _tmux('set-option','-w','-t',wid,'remain-on-exit','on')
    return Window.fetch(wid)

Like sessions, windows are created without stealing focus unless you pass `focus=True`, since a script composing a layout shouldn't yank the user's cursor around. (`remain` is a per-window setting; panes made by splitting inherit it from their window.)

In [ ]:
w = s.new_window(printer, name='work')
test_eq((w.name, w.n_panes), ('work', 1))
test_eq(s.refresh().n_wins, 2)
assert not w.active  # focus stayed put
w

<div class="prose" markdown="1">

```python
2: work (1 panes) @2
  2.1: [80x10] %2 tmux (active)
```

</div>

In [ ]:
#| export
@patch
def kill(self:Window):
    "Kill this window and its panes"
    sys.audit('fastmux.Window.kill', self.id)
    _tmux('kill-window','-t',self.id)

@patch
def rename(self:Window, name):
    "Rename this window, returning it"
    _tmux('rename-window','-t',self.id,name)
    return self.refresh()

@patch
def select(self:Window):
    "Make this the session's current window, returning it"
    _tmux('select-window','-t',self.id)
    return self.refresh()

@patch
def layout(self:Window, name):
    "Apply a tmux layout (`even-horizontal`, `tiled`, ...), returning this window"
    _tmux('select-layout','-t',self.id,name)
    return self.refresh()

In [ ]:
test_eq(w.rename('jobs').name, 'jobs')
assert w.select().active

## Splitting panes

Splitting divides a pane in two, and is how layouts get built. tmux's own flags are famously backwards (`-h` puts panes side by side; `-v` stacks them), so fastmux speaks in directions instead: where the *new* pane goes, `right`/`below`/`left`/`above`, with a one-word verb for each. Like window creation, splits never steal focus unless asked.

In [ ]:
#| export
_split_flags = dict(right=['-h'], below=['-v'], left=['-h','-b'], above=['-v','-b'])

@patch
def split(
    self:Pane,
    where='right', # Where the new pane goes: `right`, `below`, `left`, or `above`
    cmd=None,      # Command to run (str or argv list); default: the user's shell
    size=None,     # Size of the new pane: rows/columns (int) or `'30%'`
    cwd=None,      # Working directory
    env=None,      # Extra environment vars as a dict
    focus=False,   # Make the new pane active
):
    "Split this pane, returning the new `Pane`"
    args = ['split-window','-P','-F','#{pane_id}',*_split_flags[where],'-t',self.id]
    if not focus: args += ['-d']
    if size: args += ['-l',str(size)]
    if cwd: args += ['-c',str(cwd)]
    for k,v in (env or {}).items(): args += ['-e',f'{k}={v}']
    if cmd: args += [cmd] if isinstance(cmd,str) else list(cmd)
    return Pane.fetch(_tmux(*args))

@patch
@delegates(Pane.split, but=['where'])
def rsplit(self:Pane, **kwargs): return self.split('right', **kwargs)
@patch
@delegates(Pane.split, but=['where'])
def bsplit(self:Pane, **kwargs): return self.split('below', **kwargs)
@patch
@delegates(Pane.split, but=['where'])
def lsplit(self:Pane, **kwargs): return self.split('left', **kwargs)
@patch
@delegates(Pane.split, but=['where'])
def asplit(self:Pane, **kwargs): return self.split('above', **kwargs)

In [ ]:
right = p.rsplit()
above = p.asplit(size=3)
test_eq(len(Window.fetch(p.window_id).panes), 3)
assert above.height==3 and right.id != p.id
assert Pane.fetch(p.window_id).id == p.id  # focus in this window untouched
Window.fetch(p.window_id)

<div class="prose" markdown="1">

```python
1: nbs (3 panes) @0
  1.1: [40x3] %4 bash
  1.2: [40x6] %0 python (active)
  1.3: [39x10] %3 bash
```

</div>

In [ ]:
#| export
@patch
def kill(self:Pane):
    "Kill this pane (killing its window/session if it is the last one)"
    sys.audit('fastmux.Pane.kill', self.id)
    _tmux('kill-pane','-t',self.id)

@patch
def resize(self:Pane, width=None, height=None):
    "Resize this pane to absolute `width`/`height`, returning it"
    args = ['resize-pane','-t',self.id]
    if width: args += ['-x',str(width)]
    if height: args += ['-y',str(height)]
    _tmux(*args)
    return self.refresh()

@patch
def zoom(self:Pane):
    "Toggle this pane fullscreen within its window, returning it"
    _tmux('resize-pane','-Z','-t',self.id)
    return self.refresh()

@patch
def select(self:Pane):
    "Make this the active pane, returning it"
    _tmux('select-pane','-t',self.id)
    return self.refresh()

In [ ]:
test_eq(above.resize(height=5).height, 5)
assert right.select().active
above.kill(); right.kill()
test_eq(len(Window.fetch(p.window_id).panes), 1)

## Searching

Any scope can be grepped: one pane's transcript, every pane in a window or session, or every pane on the server. Matches display rg-style, and each match's `target` is a real tmux target, so you can paste it back into `tmux()` to get the pane that said it.

In [ ]:
#| export
SEARCH_LINES = 2000

class SearchMatch(AttrDict):
    "One matching transcript line, shown rg-style"
    def __repr__(self): return f'{self.target}:{self.line_no}: {self.line}'
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

class SearchResults(L):
    "Search matches, one rg-style row per line"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

def _matcher(pattern, regex=False, ignore_case=True):
    if regex: return re.compile(pattern, re.IGNORECASE if ignore_case else 0).search
    if ignore_case:
        pattern = pattern.lower()
        return lambda l: pattern in l.lower()
    return lambda l: pattern in l

def _search_panes(ps, pattern, lines=SEARCH_LINES, regex=False, ignore_case=True):
    f = _matcher(pattern, regex, ignore_case)
    return SearchResults(SearchMatch(target=c.target, line_no=c.start+i, line=l, id=c.id)
        for c in (p.display(lines) for p in ps) for i,l in enumerate(c.lines) if f(l))

@patch
def search(self:Pane, pattern, lines=SEARCH_LINES, regex=False, ignore_case=True):
    "Search this pane's recent transcript, rg-style"
    return _search_panes([self], pattern, lines, regex, ignore_case)

@patch
def search(self:Window, pattern, lines=SEARCH_LINES, regex=False, ignore_case=True):
    "Search every pane in this window, rg-style"
    return _search_panes(self.panes, pattern, lines, regex, ignore_case)

@patch
def search(self:Session, pattern, lines=SEARCH_LINES, regex=False, ignore_case=True):
    "Search every pane in this session, rg-style"
    return _search_panes(self.panes, pattern, lines, regex, ignore_case)

In [ ]:
hits = p.search('ln 2', lines=100)
test_eq(hits[0].line, 'ln 2')
test_eq(len(hits), 11)  # ln 2 and ln 20..29
hits

fastmux-demo:1.1:2: ln 2
fastmux-demo:1.1:20: ln 20
fastmux-demo:1.1:21: ln 21
fastmux-demo:1.1:22: ln 22
fastmux-demo:1.1:23: ln 23
fastmux-demo:1.1:24: ln 24
fastmux-demo:1.1:25: ln 25
fastmux-demo:1.1:26: ln 26
fastmux-demo:1.1:27: ln 27
fastmux-demo:1.1:28: ln 28
fastmux-demo:1.1:29: ln 29

In [ ]:
m = hits[0]
test_eq(m.target, p.target)
test_eq(p[m.line_no], m.line)  # line numbers are absolute transcript coordinates

## tmux()

`tmux()` with no arguments returns every session as an indented session/window/pane tree, empty if no server is running at all. With a target it returns the right live handle: a session name or `$id` gives a `Session`, `sess:win` or `@id` a `Window`, `sess:win.pane` or `%id` a `Pane`.

In [ ]:
#| export
def _list_srv(cmd, fields, *flags):
    "`_list`, with no reachable server treated as empty"
    try: return _list(cmd, fields, *flags)
    except TmuxError as e:
        if 'no server running' in str(e) or 'error connecting to' in str(e): return []
        raise

class Sessions(L):
    "Sessions shown as an indented session/window/pane tree"
    def __repr__(self):
        def _ind(o): return '\n'.join(f'  {l}' for l in repr(o).splitlines())
        return '\n'.join('\n'.join([repr(o)] + [_ind(w) for w in o.windows]) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

    def search(self, pattern, lines=SEARCH_LINES, regex=False, ignore_case=True):
        "Search every pane in every session, rg-style"
        return _search_panes([Pane(d) for d in _list_srv('list-panes', _pane_f, '-a')], pattern, lines, regex, ignore_case)

def tmux(target=None):
    "All sessions as a tree, or a live handle for `target` (session name, `sess:win`, `sess:win.pane`, or a `$`/`@`/`%` id)"
    if target is None: return Sessions(Session(d) for d in _list_srv('list-sessions', _sess_f))
    t = str(target)
    if t.startswith('$'): return Session.fetch(t)
    if t.startswith('@') or (':' in t and '.' not in t): return Window.fetch(_tmux('display-message','-p','-t',t,'#{window_id}'))
    if t.startswith('%') or ':' in t or '.' in t: return Pane.fetch(_tmux('display-message','-p','-t',t,'#{pane_id}'))
    _tmux('has-session','-t',f'={t}')  # exact-name existence check
    return Session.fetch(t)

In [ ]:
tmux()

fastmux-demo: 2 windows
  1: nbs (1 panes) @0
    1.1: [80x10] %0 python (active)
  2: jobs* (1 panes) @2
    2.1: [80x10] %2 python (active)

In [ ]:
test_eq(tmux(s.name).id, s.id)
test_eq(tmux(s.id).name, s.name)
test_eq(tmux(p.id).target, p.target)
test_eq(tmux(p.target).id, p.id)          # target strings round-trip
test_eq(tmux(p.window_id).idx, p.win)
assert any(o.id==s.id for o in tmux())
with expect_fail(TmuxError): tmux('no-such-session')

Search for a line, then jump to the pane that printed it:

In [ ]:
hit = tmux().search('ln 29')[0]
test_eq(tmux(hit.target).id, hit.id)

One more query: `current_pane` answers "which pane is *this*?". Inside tmux it is the caller's own pane; outside, tmux resolves it to the active pane of the most recently used session. `fastmux.bg` uses it to give `sid=None` its meaning.

In [ ]:
#| export
def current_pane():
    "The pane tmux considers current: the caller's own inside tmux, else the active pane of the most recent session"
    return Pane.fetch(_tmux('display-message','-p','#{pane_id}'))

In [ ]:
assert current_pane().id.startswith('%')

Finally, clean up the playground:

In [ ]:
s.kill()
with expect_fail(TmuxError, contains="can't find"): tmux('fastmux-demo')

## Export -

In [ ]:
#| hide
#| eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()